# 02 · Reproduce TANL (CVPR 2026) from the feature cache

**Acceptance (pre-G1 plan):** Four-OOD FPR95 within **±1** of **9.81**; OpenOOD v1.5 near / far FPR95
close to **60.06 / 17.21** (ViT-B/16, M = 1,000, L = 300).

**What runs** (all from cached features, 3 stream seeds):

1. equivalence of our TANL and the official postprocessor on *real* features;
2. Four-OOD, OpenOOD v1.5 near/far, and near-OOD with the 50k ID set;
3. two hyper-parameter sets — **paper** (g = 0.2, α = 0.95, as in Sec. 4.1) and **official_sh**
   (g = 0.5, α = 0, what `scripts/ood/TANL/official.sh` runs) — each in **fp32** and **fp16**
   (bit-exact emulation of the official GPU arithmetic);
4. temporal shift (Tab. A15) and ID-first / OOD-first orders (Tab. A16);
5. a worked example of modifying TANL (`TANL_LCB`, one overridden method).

**Settings:** GPU recommended (≈ 30–45 min; CPU works for fp32 only, several hours). Internet not needed.
**Inputs:** `oodlab-code` + the output of notebook 01.

In [ ]:
# ── Set-up 1/2: find the oodlab code (attached as a private Kaggle dataset) ─────────
import os, sys

def find_oodlab(roots, max_depth=4):
    """Bounded breadth-first search for a folder containing oodlab/__init__.py.
    Never walks image folders (the old notebook hung on a recursive glob over ImageNet)."""
    skip = {"ILSVRC", "train", "val", "test", "images", "__pycache__", "cache", "results", "logs"}
    for root in [r for r in roots if r and os.path.isdir(r)]:
        frontier = [(root, 0)]
        while frontier:
            d, depth = frontier.pop(0)
            if os.path.isfile(os.path.join(d, "oodlab", "__init__.py")):
                return d
            if depth < max_depth:
                try:
                    subs = sorted(e.path for e in os.scandir(d)
                                  if e.is_dir(follow_symlinks=False) and e.name not in skip)
                except OSError:
                    continue
                frontier += [(s, depth + 1) for s in subs]
    return None

ROOTS = [os.environ.get("OODLAB_CODE"), "/kaggle/input", "/kaggle/working", os.getcwd()]
CODE_ROOT = find_oodlab(ROOTS)
if CODE_ROOT is None:   # the dataset was uploaded as a zip that Kaggle did not unpack -> unpack it ourselves
    import zipfile
    for root in [r for r in ROOTS[:2] if r and os.path.isdir(r)]:
        for dp, dn, fns in os.walk(root):
            dn[:] = [d for d in dn if d not in {"ILSVRC", "train", "val", "test", "images"}] if dp.count(os.sep) - root.count(os.sep) < 3 else []
            for fn in fns:
                if fn.startswith("oodlab") and fn.endswith(".zip"):
                    zipfile.ZipFile(os.path.join(dp, fn)).extractall("/kaggle/working/_oodlab_code")
    CODE_ROOT = find_oodlab(["/kaggle/working/_oodlab_code"])
if CODE_ROOT is None:
    raise FileNotFoundError("oodlab code not found: attach your private dataset with the code "
                            "(Add Input -> Your Datasets -> oodlab-code). See GUIDE.md, section 2.")
if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)
import oodlab
print("oodlab", oodlab.__version__, "loaded from", CODE_ROOT)

In [ ]:
# ── Set-up 2/2: packages, folders, logging ─────────────────────────────────────────
from oodlab.session import start, finish, ensure_packages
ensure_packages(clip=True, gdown=False)      # CLIP ships inside oodlab; this only adds small pip packages if missing
ctx = start("02_reproduce_tanl", need_gpu=False)
log = ctx.log                                   # log.info(...) -> screen + /kaggle/working/logs/02_reproduce_tanl.log
import numpy as np, pandas as pd, torch, time, json
from oodlab.report import show, save_tables, write_report

### Settings

In [ ]:
from oodlab.methods import TANL, TANLConfig
from oodlab.runner import Runner, summarize, compare_with_targets, acceptance_table
from oodlab.report import find_cache

BACKBONE = "ViT-B/16"
SEEDS = [0, 1, 2]
CONFIGS = {"paper": TANLConfig.paper(), "official_sh": TANLConfig.official_sh()}
PRECISIONS = ["fp32", "fp16"] if ctx.device == "cuda" else ["fp32"]   # fp16 emulation is slow on CPU
RUN = dict(equivalence=True, four_ood=True, openood_v15=True, near_50k=True, temporal=True, orders=True, variant_demo=True)
STREAM_CONFIG = "auto"           # config for temporal / order runs; "auto" = whichever matches 9.81 best
if ctx.smoke:
    SEEDS = [0, 1]
    CONFIGS = {k: v.with_(num_neg=100) for k, v in CONFIGS.items()}

In [ ]:
cache = find_cache(ctx, BACKBONE)
bank = cache.load_textbank()
LS = bank.logit_scale
print(json.dumps(bank.summary(), indent=1))
runner = Runner(cache, device=ctx.device, logger=log, keep_orders=True)

def make_tanl(cfg, precision, cls=TANL, **kw):
    c = cfg.with_(logit_scale=LS)
    if precision == "fp16":
        c = c.with_(emulate_fp16=True, exact_fp16_final=True)
    return cls(bank.id_text, bank.corpus_text, bank.noise_feats, c, device=ctx.device, n_neglabel=bank.n_selected, **kw)

runs = []   # every per-(dataset, seed) row goes here

### 1 · Our TANL == official TANL on real features

Same stream (4,000 ImageNet + 2,000 iNaturalist images) through the vendored official postprocessor and
through our class. Expect max |Δscore| ≈ 1e-6 in fp32 and exactly 0 in fp16.

In [ ]:
eq_sets = [n for n in ["inaturalist", "ninco", "sun", "places", "textures_all", "ssb_hard", "openimage_o"] if cache.has(n)]
if RUN["equivalence"] and cache.has("imagenet_val_all") and eq_sets:
    from oodlab.official.equivalence import compare_tanl
    from oodlab.stream import pair_stream
    ids, ood = cache.load_set("imagenet_val_all"), cache.load_set(eq_sets[0])
    n_id, n_ood = min(4000, len(ids)), min(2000, len(ood))
    s = pair_stream(ids.subset(range(n_id)), ood.subset(range(n_ood)), seed=0)
    batches = [s.feats(a, b) for _, a, b, _ in s.batches()]
    cfg = CONFIGS["paper"].with_(logit_scale=LS)
    eq = {"fp32": compare_tanl(bank.id_text.float(), bank.corpus_text.float(), bank.noise_feats.float(),
                               [b.float() for b in batches], bank.n_selected, cfg, device=ctx.device)}
    if "fp16" in PRECISIONS:
        eq["fp16"] = compare_tanl(bank.id_text.half(), bank.corpus_text.half(), bank.noise_feats.half(),
                                  [b.half() for b in batches], bank.n_selected,
                                  cfg.with_(emulate_fp16=True, exact_fp16_final=True), device=ctx.device)
    show(pd.DataFrame(eq).T, f"official vs ours (real features: ImageNet + {eq_sets[0]})", digits=8)
    ok32 = eq["fp32"]["max_abs_diff"] < 1e-4 and eq["fp32"]["pred_agreement"] >= 0.999
    ok16 = "fp16" not in eq or (eq["fp16"]["max_abs_diff"] <= 2e-3 and eq["fp16"]["pred_agreement"] >= 0.999)
    if ok32 and ok16:
        print("EQUIVALENCE OK", "(fp16 bit-identical)" if eq.get("fp16", {}).get("max_abs_diff", 1) == 0 else "")
    else:
        log.warning("EQUIVALENCE CHECK FAILED - results below are still computed, but investigate before trusting them")
    oodlab.utils.write_json(eq, ctx.path("equivalence_real_features.json"))
else:
    print("equivalence check skipped (needs imagenet_val_all and one OOD set in the cache)")

### 2 · Benchmarks

In [ ]:
for protocol in ["four_ood", "openood_v15", "near_50k"]:
    if not RUN[protocol]:
        continue
    miss = runner.missing(protocol)
    if miss:
        log.warning(f"skip {protocol}: sets not cached yet {miss}")
        continue
    for cname, cfg in CONFIGS.items():
        for prec in PRECISIONS:
            runs.append(runner.evaluate(make_tanl(cfg, prec), protocol, SEEDS, config=f"{cname}/{prec}"))
if not runs:
    raise RuntimeError(f"no protocol could run - cached sets: {cache.available()}; finish notebook 01 first")
df = pd.concat(runs, ignore_index=True)
summary = summarize(df)
show(summary[summary.key.str.startswith("mean:")],
     "group means (FPR95 lower is better)", cols=["config", "protocol", "key", "fpr95", "fpr95_std", "auroc", "fpr95_idpos", "acc"])

In [ ]:
for protocol in ["four_ood", "openood_v15"]:
    show(compare_with_targets(summary, "tanl", protocol), f"TANL vs paper: {protocol}")
acc_table = acceptance_table(summary)
show(acc_table, "pre-G1 acceptance (TANL)")

### 3 · Temporal shift (Tab. A15) and sample order (Tab. A16)

Temporal shift: the four OOD sets arrive one after another (each mixed with the full ID set) and
the queues are **not** reset between them. Sample order: all ID first, or all OOD first.

In [ ]:
from oodlab.report import best_tanl_config
stream_rows = []
best_prec = "fp16" if "fp16" in PRECISIONS else "fp32"
if STREAM_CONFIG == "auto":
    STREAM_CONFIG = best_tanl_config(summary, precision=best_prec) or "paper"
    log.info(f"temporal / order experiments use the '{STREAM_CONFIG}' configuration")
if RUN["temporal"] and not runner.missing("four_ood"):
    t = runner.evaluate_temporal(make_tanl(CONFIGS[STREAM_CONFIG], best_prec), seeds=SEEDS,
                                 config=f"{STREAM_CONFIG}/{best_prec}")
    stream_rows.append(t)
if RUN["orders"] and not runner.missing("four_ood"):
    for order in ["id_first", "ood_first"]:
        stream_rows.append(runner.evaluate(make_tanl(CONFIGS[STREAM_CONFIG], best_prec), "four_ood", SEEDS,
                                           config=f"{STREAM_CONFIG}/{best_prec}", order=order, tag=f"order:{order}"))
if stream_rows:
    sdf = pd.concat(stream_rows, ignore_index=True)
    ssum = summarize(sdf)
    for protocol in sorted(ssum.protocol.unique()):
        show(compare_with_targets(ssum, "tanl", protocol), f"TANL vs paper: {protocol}")
    show(sdf[sdf.protocol.str.startswith("temporal")].groupby(["protocol", "position", "dataset"])[["fpr95", "post_shift_fpr95"]].mean().reset_index(),
         "temporal shift: whole-segment FPR95 and FPR95 on the first 2,560 samples after each shift")
else:
    sdf, ssum = pd.DataFrame(), pd.DataFrame()

### 4 · Modifying TANL: a worked example

Every step of TANL is a method you can override (`activation`, `select`, `score`, `threshold`, `admit`).
`oodlab/methods/variants.py` contains `TANL_LCB`, which ranks labels by *activation difference − κ·SE*
(a lower confidence bound) instead of the plain difference — about ten lines. `κ = 0` is exactly TANL.
Write your own variant the same way, in that file or right here in a cell.

In [ ]:
if RUN["variant_demo"] and not runner.missing("four_ood"):
    from oodlab.methods.variants import TANL_LCB
    demo = [runner.evaluate(make_tanl(CONFIGS["paper"], "fp32", cls=TANL_LCB, kappa=k), "four_ood", SEEDS[:1],
                            config=f"paper/fp32/lcb{k}") for k in [0.0, 1.0]]
    demo_sum = summarize(pd.concat(demo, ignore_index=True))
    show(demo_sum[demo_sum.key == "mean:four"], "TANL_LCB example (seed 0)", cols=["method", "config", "fpr95", "auroc"])

### 5 · Save

In [ ]:
all_df = pd.concat([df] + ([sdf] if len(sdf) else []), ignore_index=True)
all_sum = summarize(all_df)
save_tables(ctx, tanl_runs=all_df, tanl_summary=all_sum, tanl_acceptance=acc_table)
runner.save_orders(ctx.path("stream_orders.npz"))
sections = [("Acceptance", acc_table)]
for p in ["four_ood", "openood_v15", "near_50k"] + sorted(ssum.protocol.unique() if len(ssum) else []):
    c = compare_with_targets(all_sum, "tanl", p)
    if len(c):
        sections.append((f"TANL vs published: {p}", c))
sections.append(("All group means", all_sum[all_sum.key.str.startswith("mean:")][
    ["config", "protocol", "key", "fpr95", "fpr95_std", "auroc", "auroc_std", "fpr95_idpos", "acc", "n_seeds"]]))
write_report(ctx, "TANL reproduction", sections)
finish(ctx, {"acceptance": acc_table.to_dict("records")})